In [92]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [93]:
import sys
import torch, torch.nn as nn, torch.optim as optim
from pathlib import Path
project_root = Path.cwd().resolve().parents[2]
sys.path.append(str(project_root))

from defs.vae.ae import *
from defs.vae.training_defs import *

In [94]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

spectra, fractions = load_spectral_data()
train_loader, val_loader = get_dataloaders(spectra, fractions)
print(spectra.shape)

Device: cpu
(1723, 210)


In [ ]:
convolution = ConvEncoder(spectra.shape[1], 128, spectra.shape[1], layers=1, kernel_size=4)
criterion = nn.MSELoss()
optimizer = optim.Adam(convolution.parameters(), lr=1e-3)

print(convolution.encoder)

n_epochs = 1
train_losses, val_losses = [], []

Sequential(
  (0): Conv1d(210, 128, kernel_size=(4,), stride=(1,), padding=(1,))
  (1): ReLU()
  (2): MaxPool1d(kernel_size=3, stride=1, padding=0, dilation=1, ceil_mode=False)
  (3): Conv1d(128, 128, kernel_size=(4,), stride=(1,), padding=(1,))
  (4): ReLU()
  (5): MaxPool1d(kernel_size=3, stride=1, padding=0, dilation=1, ceil_mode=False)
  (6): Conv1d(128, 210, kernel_size=(4,), stride=(1,), padding=(1,))
  (7): ReLU()
  (8): MaxPool1d(kernel_size=3, stride=1, padding=0, dilation=1, ceil_mode=False)
)


In [96]:
for epoch in range(1, n_epochs+1):
    convolution.train()
    tot_train = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss_pred = criterion(convolution(xb), yb)
        loss_pred.backward()
        optimizer.step()

        loss = loss_pred
        tot_train += loss.item() * xb.size(0)
    train_losses.append(tot_train / len(train_loader.dataset))

    convolution.eval()
    tot_val = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            tot_val += criterion(convolution(xb), yb).item() * xb.size(0)
    val_losses.append(tot_val / len(val_loader.dataset))

    print(f"epoch {epoch:02d}  train loss: {train_losses[-1]:.4f}  val loss: {val_losses[-1]:.4f}")

RuntimeError: Given groups=1, weight of size [128, 210, 4], expected input[1, 32, 210] to have 210 channels, but got 32 channels instead

In [ ]:
#Thanks Ege


"""
# Testing the graph of the epsilon

# Create the test tensors
spec_tensor = torch.randn(5, 81).to(device)
time_tensor = torch.randn(5,).to(device)
ab_tensor = torch.randn(5, 3).to(device)

output = convolution()


# Create the graph:
from torchviz import make_dot
import graphviz
import os
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"

make_dot(output, params=dict(convolution.named_parameters())).render("model_graph", format="png")
"""